In [ ]:
"""
Minimal "fetch-only" examples for:
1) RemoteOK (public JSON API, no key)
2) We Work Remotely (RSS feed, no key)
3) SerpAPI -> Google Jobs (requires SERPAPI_API_KEY)

This script only calls and prints raw responses (or a short preview).
You can parse/process however you like.
"""

import os
import sys
import json
import urllib.parse
import requests
import requests
import xml.etree.ElementTree as ET
import json
from dotenv import load_dotenv

from pydantic import BaseModel, computed_field
from functools import cached_property
from typing import Optional
import tiktoken

from IPython.core.display import display, HTML
from bs4 import BeautifulSoup
import html
import re

from openai import OpenAI

C:\Users\prash_rppt1\AppData\Local\Temp\ipykernel_40124\1679550918.py:21: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


In [2]:
load_dotenv(override=True)  # Load .env variables, override existing ones if any

SERPAPI_API_KEY = os.getenv("SERPAPI_KEY")
OPEN_AI_KEY = os.getenv("OPENAI_API_KEY")

embedding_model = "text-embedding-3-small"

In [ ]:
import numpy as np

openai_client = OpenAI(api_key=OPEN_AI_KEY)

def embed(text: str) -> list[float]:
    '''Get embedding vector for a single text using OpenAI API'''
    response = openai_client.embeddings.create(
        input=text,
        model=embedding_model,
    )
    return response.data[0].embedding


def embed_jobs(jobs: list, batch_size: int = 500) -> None:
    '''Embed all jobs in batches, storing the result in job.embedding in-place'''
    for i in range(0, len(jobs), batch_size):
        batch = jobs[i:i + batch_size]
        texts = [job.embedding_text for job in batch]

        response = openai_client.embeddings.create(
            input=texts,
            model=embedding_model,
        )

        ordered = sorted(response.data, key=lambda x: x.index)
        for job, result in zip(batch, ordered):
            job.embedding = result.embedding

        print(f"Embedded {min(i + batch_size, len(jobs))}/{len(jobs)} jobs")


def score_jobs(jobs: list, profile_embedding: list[float]) -> None:
    '''Embed all jobs then compute cosine similarity against profile_embedding in-place.
    Since OpenAI embeddings are unit-normalized, cosine similarity == dot product.'''
    embed_jobs(jobs)

    profile_vec = np.array(profile_embedding)
    job_vecs = np.array([job.embedding for job in jobs])  # shape: (n_jobs, 1536)
    scores = job_vecs @ profile_vec                        # shape: (n_jobs,)

    for job, score in zip(jobs, scores):
        job.similarity_score = float(score)
    
    print("Scored all jobs against profile embedding")

### llm-evaluation functions

In [62]:
EVALUATOR_SYSTEM_PROMPT = """You are evaluating job postings to decide if they are suitable for a candidate.

Evaluate the job strictly against these three criteria:

1. REMOTE & LOCATION (is_truly_remote):
   - PASS if the job is worldwide remote, or if no location restriction is mentioned.
   - FAIL if the job says "remote" but restricts to a specific country, state, or city
     (e.g. "Remote - US only", "must reside in Canada", "Remote - NYC area").
   - Working from India must be possible. When in doubt, lean PASS.

2. ROLE RELEVANCE (is_relevant_role):
   - PASS for genuine technical AI/ML/Data Science roles: AI Engineer, ML Engineer,
     LLM Engineer, Applied Scientist, NLP Engineer, Data Scientist, MLOps, etc.
   - FAIL if the role is only tangentially related: AI data annotator, AI content
     reviewer, AI sales engineer, AI product manager with no hands-on technical work.

3. CONTRACT DURATION (is_long_term):
   - PASS if the role is permanent, long-term, or if duration is not mentioned.
   - FAIL only if explicitly stated as a short contract of 8 weeks or less
     (e.g. "6-week engagement", "2-month project").

Set overall suitable=True only if ALL three criteria pass.
Be concise in reasoning — one sentence per criterion is enough."""


class JobEvaluation(BaseModel):
    is_truly_remote: bool
    is_relevant_role: bool
    is_long_term: bool
    suitable: bool
    reasoning: str


def evaluate_single_job_llm(job: Job) -> JobEvaluation:
    '''Evaluate a single job using the LLM and return a structured JobEvaluation result.'''
    user_content = f"""Job Title: {job.title}
Company: {job.company}
Location: {job.location or 'Not specified'}

{job.description}"""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        response_format=JobEvaluation,
    )
    return response.choices[0].message.parsed


def evaluate_jobs_with_llm(jobs: list[Job], max_workers: int = 10) -> None:
    """Run LLM evaluation on all jobs in parallel."""
    from tqdm import tqdm
    from concurrent.futures import ThreadPoolExecutor, as_completed

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_job = {executor.submit(evaluate_single_job_llm, job): job for job in jobs}

        for future in tqdm(as_completed(future_to_job), total=len(jobs)):
            job = future_to_job[future]
            job.llm_evaluation = future.result()

    print(f"Evaluated {len(jobs)} jobs with LLM.")

In [4]:
def google_job_to_str(job_dct):
    google_job_str = f'''job_title: {job_dct.get('job_title', 'not available')}
    company_name: {job_dct.get('company_name', 'not available')}

    Job Description: {job_dct.get('description', 'not available')}

    Job Highlights: {job_dct.get('job_highlights', 'not available')}

    extensions: {job_dct.get('detected_extensions', 'not available')}

    job_location: {job_dct.get('job_location', 'not available')}
    '''

    return(google_job_str)

# function to display html
def _display_html(html):
    display(HTML(html))


def _html_to_clean_text(raw_html: str) -> str:
    if not raw_html:
        return ""

    # 1. Decode HTML entities (&amp;, &nbsp;, â etc.)
    decoded = html.unescape(raw_html)

    # 2. Parse HTML
    soup = BeautifulSoup(decoded, "html.parser")

    # 3. Remove scripts/styles just in case
    for tag in soup(["script", "style"]):
        tag.decompose()

    # 4. Get text with newlines preserved
    text = soup.get_text(separator="\n")

    # 5. Normalize whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)   # collapse excessive newlines
    text = re.sub(r"[ \t]+", " ", text)      # collapse spaces
    text = text.strip()

    return text

In [5]:
def fetch_remoteok(timeout=30):
    url = "https://remoteok.com/api"
    headers = {
        # Be polite; many sites prefer a real UA
        "User-Agent": "pj-job-fetcher/0.1 (+https://example.local)",
        "Accept": "application/json",
    }
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    d = r.json()
    data = []
    for job in d:
        if 'legal' in job:
            continue
        job['description_html'] = job.get('description', '')
        job['description'] = _html_to_clean_text(job['description_html'])
        data.append(job)    
    return data


def fetch_wwr_rss_as_json(timeout=30):
    rss_url = "https://weworkremotely.com/categories/remote-programming-jobs.rss"
    r = requests.get(                    
        rss_url,
        headers={"User-Agent": "pj-job-fetcher/0.1"},
        timeout=timeout,
    )
    r.raise_for_status()

    root = ET.fromstring(r.text)

    channel = root.find("channel")
    if channel is None:
        return {"raw": r.text}

    items = []
    for item in channel.findall("item"):
        items.append({
            "title": (item.findtext("title") or "").strip(),
            "link": (item.findtext("link") or "").strip(),
            "guid": (item.findtext("guid") or "").strip(),
            "pubDate": (item.findtext("pubDate") or "").strip(),
            "description_html": (item.findtext("description") or "").strip(),
            "description": _html_to_clean_text(item.findtext("description") or ""),
        })

    return items


def fetch_serpapi_google_jobs(
    query="remote ai llm engineer",
    hl="en",
    gl="us",
    max_pages=5,
    timeout=30,
    add_provenance=True,
    retry=1,
):
    '''
    Fetch Google Jobs results via SerpAPI.
    Args:
        query: Search query string.
        hl: Language (e.g. "en").
        gl: Country (e.g. "us") or list of countries to search.
        max_pages: Max number of pages to fetch per country.
        timeout: Request timeout in seconds.
        add_provenance: If True, adds "_gl" and "_query" fields to each job result for provenance.
        retry: Number of retries for transient errors (429, 5xx).
    Returns:
        List of job results (dicts) from SerpAPI.
    '''
    if not SERPAPI_API_KEY:
        raise RuntimeError("Missing SERPAPI_API_KEY env var. Example: export SERPAPI_API_KEY='...'")

    if max_pages < 1:
        return []

    base = "https://serpapi.com/search.json"
    headers = {
        "User-Agent": "pj-job-fetcher/0.1 (+https://example.local)",
        "Accept": "application/json",
    }

    if isinstance(gl, str):
        gl_list = [gl]
    else:
        gl_list = list(gl)

    if not gl_list:
        return []

    all_jobs = []

    for gl_ in gl_list:
        params = {
            "engine": "google_jobs",
            "q": query,
            "hl": hl,
            "gl": gl_,
            "api_key": SERPAPI_API_KEY,
        }

        next_page_token = None
        seen_tokens = set()

        for _page in range(1, max_pages + 1):
            if next_page_token:
                # prevent token loops
                if next_page_token in seen_tokens:
                    break
                seen_tokens.add(next_page_token)
                params["next_page_token"] = next_page_token
            else:
                params.pop("next_page_token", None)

            url = f"{base}?{urllib.parse.urlencode(params)}"

            last_err = None
            for attempt in range(retry + 1):
                try:
                    r = requests.get(url, headers=headers, timeout=timeout)
                    r.raise_for_status()
                    data = r.json()
                    last_err = None
                    break
                except requests.HTTPError as e:
                    last_err = e
                    status = getattr(e.response, "status_code", None)
                    # simple retry on rate limit / transient server errors
                    if attempt < retry and status in (429, 500, 502, 503, 504):
                        time.sleep(1.5 * (attempt + 1))
                        continue
                    raise
            if last_err:
                raise last_err

            jobs = data.get("jobs_results", []) or []
            if add_provenance:
                for j in jobs:
                    if isinstance(j, dict):
                        j["_gl"] = gl_
                        j["_query"] = query
            all_jobs.extend(jobs)

            pagination = data.get("serpapi_pagination", {}) or {}
            next_page_token = pagination.get("next_page_token")

            if not next_page_token:
                break

    return all_jobs

## Fetch jobs

In [6]:
try:
    # Fetch RemoteOK jobs
    remoteok_jobs = fetch_remoteok()
    print(f"RemoteOK: Fetched {len(remoteok_jobs)} jobs.")
except Exception as e:
    print(f"Error fetching RemoteOK: {e}")

try:
    # Fetch We Work Remotely RSS jobs
    wwr_jobs = fetch_wwr_rss_as_json()
    print(f"We Work Remotely RSS: Fetched {len(wwr_jobs)} jobs.")
except Exception as e:
    print(f"Error fetching We Work Remotely RSS: {e}")

RemoteOK: Fetched 97 jobs.
We Work Remotely RSS: Fetched 25 jobs.


In [7]:
QUERIES = [
    'remote ("AI Engineer" OR "Machine Learning Engineer" OR "Applied AI Engineer")',
    'remote ("LLM" OR "Large Language Model" OR "Generative AI" OR "GenAI" OR "NLP")',
    'remote ("LangChain" OR "LlamaIndex" OR "RAG" OR "OpenAI API" OR "GPT")',
    'remote ("LLM deployment" OR "model serving" OR "inference" OR "ML Platform" OR "MLOps")',
]

GLS = ["us", "ca", "gb", "de", "nl", "sg", "au"]

google_jobs = []
for q in QUERIES:
    try:
        jobs = fetch_serpapi_google_jobs(query=q, gl=GLS, max_pages=5)
        google_jobs.extend(jobs)
        print(f"Google Jobs: Fetched {len(jobs)} jobs for query '{q}'.")
    except Exception as e:
        print(f"Error fetching Google Jobs for query '{q}': {e}")

print(f"Total Google Jobs fetched: {len(google_jobs)}")

Google Jobs: Fetched 42 jobs for query 'remote ("AI Engineer" OR "Machine Learning Engineer" OR "Applied AI Engineer")'.
Google Jobs: Fetched 0 jobs for query 'remote ("LLM" OR "Large Language Model" OR "Generative AI" OR "GenAI" OR "NLP")'.
Google Jobs: Fetched 0 jobs for query 'remote ("LangChain" OR "LlamaIndex" OR "RAG" OR "OpenAI API" OR "GPT")'.
Google Jobs: Fetched 0 jobs for query 'remote ("LLM deployment" OR "model serving" OR "inference" OR "ML Platform" OR "MLOps")'.
Total Google Jobs fetched: 42


## Unified job model

In [ ]:
_encoding = tiktoken.get_encoding("cl100k_base")  # encoding used by text-embedding-3-small


class Job(BaseModel):
    title: str
    company: str
    description: str
    salary: Optional[str]
    date: Optional[str]
    location: Optional[str]
    link: str
    source: str                       # "remoteok" | "wwr" | "google"
    tags: Optional[list[str]]         # available from RemoteOK; None for others
    embedding: Optional[list[float]] = None
    similarity_score: Optional[float] = None  # to be filled in after scoring
    llm_evaluation: Optional[JobEvaluation] = None  # to be filled in after LLM evaluation

    model_config = {"arbitrary_types_allowed": True}  # required for cached_property

    @computed_field
    @cached_property
    def embedding_text(self) -> str:
        parts = [f"Job Title: {self.title}"]
        if self.company:
            parts.append(f"Company: {self.company}")
        if self.tags:
            parts.append(f"Tags: {', '.join(self.tags)}")
        parts.append(f"\n{self.description}")
        return "\n".join(parts)

    @computed_field
    @cached_property
    def token_count(self) -> int:
        return len(_encoding.encode(self.embedding_text))


def extract_job(job_dict: dict, source: str) -> Job:
    if source == "remoteok":
        salary_min = job_dict.get("salary_min", 0)
        salary_max = job_dict.get("salary_max", 0)
        salary = f"${salary_min}–${salary_max}" if (salary_min or salary_max) else None

        return Job(
            title=job_dict.get("position", ""),
            company=job_dict.get("company", ""),
            description=job_dict.get("description", ""),
            salary=salary,
            date=job_dict.get("date"),
            location=job_dict.get("location"),
            link=job_dict.get("url", ""),
            source="remoteok",
            tags=job_dict.get("tags"),
        )

    elif source == "wwr":
        raw_title = job_dict.get("title", "")
        if ": " in raw_title:
            company, title = raw_title.split(": ", 1)
        else:
            company, title = "", raw_title

        return Job(
            title=title,
            company=company,
            description=job_dict.get("description", ""),
            salary=None,
            date=job_dict.get("pubDate"),
            location=None,
            link=job_dict.get("link", ""),
            source="wwr",
            tags=None,
        )

    elif source == "google":
        apply_options = job_dict.get("apply_options") or []
        link = apply_options[0]["link"] if apply_options else job_dict.get("share_link", "")

        extensions = job_dict.get("extensions") or []
        salary = next(
            (e for e in extensions if any(c in e for c in ("$", "£", "€", "¥")) or "per year" in e.lower()),
            None,
        )

        return Job(
            title=job_dict.get("title", ""),
            company=job_dict.get("company_name", ""),
            description=job_dict.get("description", ""),
            salary=salary,
            date=None,
            location=job_dict.get("location"),
            link=link,
            source="google",
            tags=None,
        )

    else:
        raise ValueError(f"Unknown source: {source!r}. Expected 'remoteok', 'wwr', or 'google'.")

## processing jobs

In [47]:
combined_jobs = []

for job_dict in remoteok_jobs:
    try:
        job = extract_job(job_dict, source="remoteok")
        combined_jobs.append(job)
    except Exception as e:
        print(f"Error extracting RemoteOK job: {e}")

for job_dict in wwr_jobs:
    try:
        job = extract_job(job_dict, source="wwr")
        combined_jobs.append(job)
    except Exception as e:
        print(f"Error extracting WWR job: {e}")

for job_dict in google_jobs:
    try:
        job = extract_job(job_dict, source="google")
        combined_jobs.append(job)
    except Exception as e:
        print(f"Error extracting Google job: {e}")

### deduplication
remove duplicate jobs from the list and keep only one instance

In [48]:
# remove the ones with same url
seen_links = {}

unique_jobs = []
for job in combined_jobs:
    if job.link not in seen_links:
        seen_links[job.link] = job.source
        unique_jobs.append(job)
    else:
        print(f"Duplicate job found: {job.title} at {job.company}. Already seen from source: {seen_links[job.link]}.")

In [49]:
from collections import defaultdict

# cluster jobs by companies to further deduplicate
company_clusters = defaultdict(list)

for job in unique_jobs:
    company_clusters[job.company].append(job)

companies_with_multiple_jobs = set()
for company, jobs in company_clusters.items():
    if len(jobs) > 1:
        companies_with_multiple_jobs.add(company)
        print(f"Company '{company}' has {len(jobs)} job listings.")

Company 'Whatnot' has 4 job listings.
Company 'Accenture Federal Services' has 2 job listings.
Company '10x Team' has 2 job listings.
Company 'Eleventh Hour Games' has 6 job listings.
Company 'Cast AI' has 2 job listings.
Company 'ZÃ³calo Health' has 2 job listings.
Company 'DeweyLearn Inc.' has 2 job listings.
Company 'AHU Technologies' has 2 job listings.
Company 'Jobgether' has 2 job listings.
Company 'Dusk Labs' has 2 job listings.
Company 'Proxify AB' has 3 job listings.
Company 'Daice Labs' has 2 job listings.
Company 'Flexionis' has 3 job listings.
Company 'SimpliSafe' has 2 job listings.


In [50]:
with open("profile_summary.txt", "r") as f:
    profile_summary = f.read()

profile_embedding = embed(profile_summary)
score_jobs(unique_jobs, profile_embedding=profile_embedding)

Embedded 164/164 jobs


After some manual inspection, I can say that we can go with jobs with similarity score >.45. 

This is a rather safe threshold which may include a few irrelevant jobs but it will reduce the risk of missing any relevant opportunity.

In [51]:
filtered_jobs = [job for job in unique_jobs if job.similarity_score is not None and job.similarity_score > 0.45]
len(filtered_jobs)

83

## LLM evaluation filter

In [63]:
evaluate_jobs_with_llm(filtered_jobs)

100%|██████████| 83/83 [00:20<00:00,  3.99it/s]

Evaluated 83 jobs with LLM.


In [78]:
final_list = [job for job in filtered_jobs if job.llm_evaluation and job.llm_evaluation.model_dump()['suitable']]

In [79]:
len(final_list)

33

In [85]:
job.llm_evaluation.model_dump()

{'is_truly_remote': False,
 'is_relevant_role': True,
 'is_long_term': True,
 'suitable': False,
 'reasoning': "The job mentions a hybrid work model requiring in-office attendance, failing the remote criterion; it is a relevant technical role in applied research; and the duration isn't specified as short-term."}

## pushing to google sheet

In [86]:
import importlib
import sheets
importlib.reload(sheets)
from sheets import push_jobs_to_sheet, add_to_csv

In [87]:
push_jobs_to_sheet(final_list, sheet_name="final_list")

Added 33 new jobs to the sheet. (0 duplicates skipped)


In [88]:
push_jobs_to_sheet(filtered_jobs, sheet_name="shortlisted")

Added 83 new jobs to the sheet. (0 duplicates skipped)
